# Run COLMAP on Training Images + Trace the Camera Registration Order

This notebook runs the **actual COLMAP reconstruction pipeline** (feature extraction,
matching, incremental Structure-from-Motion) on real training images from
`dataset/`, instead of only visualizing the `.ply`/`cameras.json` that a *previous*
COLMAP run already produced (as `gaussian_point_cloud_visualization.ipynb` does for
`pipeline-results/`). It uses [pycolmap](https://github.com/colmap/colmap) (COLMAP's
official Python bindings, `pip install pycolmap`) rather than building the C++ CLI
from the `colmap/` source tree — that source is upstream COLMAP itself and a from
-source build needs CMake + vcpkg + a full C++ toolchain, which is unnecessary when a
prebuilt wheel exists for this exact purpose.

**The specific thing this notebook adds that a normal COLMAP run doesn't give you for
free**: incremental SfM registers images ONE AT A TIME, in an order it decides for
itself (starting from a well-matched seed pair, then greedily adding whichever
next image has the most confirmed 2D-3D correspondences) -- not simply the order the
files are named. That per-step order is exactly what you asked to have saved for
re-tracing/replaying later. COLMAP's C++ pipeline exposes exactly two callback hooks
for this (found by reading `colmap/src/colmap/controllers/incremental_pipeline.cc`):
`INITIAL_IMAGE_PAIR_REG_CALLBACK` (fires once, for the seed pair) and
`NEXT_IMAGE_REG_CALLBACK` (fires once per image after that). This notebook registers
Python callbacks on both, and at each firing records which image just got registered,
how many are registered so far, how many 3D points exist so far, and the elapsed time
-- exactly enough to **replay** the incremental process step by step in the browser
viewer this thesis already built (`viz/index.html`'s new "COLMAP registration replay"
slider/play control, added alongside this notebook).

**A real CPU-only timing data point from validating this notebook** (no CUDA on this
machine -- `pycolmap.has_cuda` is `False`): 40 real photos at 5187x3361 downscaled to
1200px on the long side took ~90s for feature extraction, ~270s for sequential
matching, ~85s for incremental mapping -- **about 7.5 minutes total for 40 images**.
Scale roughly linearly-to-worse with image count (matching cost grows with the
overlap window, not strictly linearly). Running a FULL scene (185-280 images) from
this thesis's `dataset/` folder would take considerably longer on CPU alone -- if you
need that, run this same notebook on a GPU machine (`pycolmap`'s SIFT extractor/
matcher both support `Device.cuda`) or Kaggle, matching this whole project's existing
practice for anything training-scale.

## Config -- edit these

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
DATASET_ROOT = REPO_ROOT / "dataset"
VIZ_ROOT = REPO_ROOT / "viz"
DATA_DIR = VIZ_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# EDIT ME
# ============================================================
SCENE_IMAGES_DIR = DATASET_ROOT / "mipnerf360" / "garden" / "images"
NUM_IMAGES = 40          # consecutive subsequence -- see the note below on why
MAX_IMAGE_SIZE = 1200    # downscale before SIFT extraction; smaller = faster, less precise
DATASET_KEY = "colmap_live_garden_40"
DATASET_LABEL = f"COLMAP live run: garden ({NUM_IMAGES} imgs)"
# ============================================================

WORK_DIR = REPO_ROOT / "_colmap_work" / DATASET_KEY
IMAGE_DIR = WORK_DIR / "images"
DB_PATH = WORK_DIR / "database.db"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

assert SCENE_IMAGES_DIR.is_dir(), f"not found: {SCENE_IMAGES_DIR}"
print("scene:", SCENE_IMAGES_DIR)
print("work dir:", WORK_DIR)

## 1. Pick a *consecutive* subsequence of images -- not an even spread

Real captures like these are taken walking around the scene: neighboring
FILENAMES are neighboring VIEWPOINTS with high visual overlap, which is exactly what
sequential feature matching assumes. **Validated the hard way** while building this
notebook: an evenly-spread subsample across the whole capture (e.g. every 6th image
of 185) breaks that overlap assumption -- the reconstruction repeatedly failed to
grow past 2-4 registered images ("Could not register, trying another image" /
"Discarding reconstruction due to insufficient size"). A plain *consecutive* run of
frames fixed it immediately (grew cleanly to all 40/40 registered, 29k+ points).

In [ ]:
import shutil

all_images = sorted(p.name for p in SCENE_IMAGES_DIR.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
print(f"{len(all_images)} images available in {SCENE_IMAGES_DIR.name}")
assert NUM_IMAGES <= len(all_images)

selected = all_images[:NUM_IMAGES]
for f in selected:
    dst = IMAGE_DIR / f
    if not dst.exists():
        shutil.copy(SCENE_IMAGES_DIR / f, dst)
print(f"staged {len(selected)} consecutive images -> {IMAGE_DIR}")

## 2. Feature extraction + sequential matching

In [ ]:
import time
import pycolmap

print("has_cuda:", pycolmap.has_cuda, "-- extraction/matching will run on CPU if False")

if DB_PATH.exists():
    DB_PATH.unlink()

extraction_options = pycolmap.FeatureExtractionOptions(max_image_size=MAX_IMAGE_SIZE)

t0 = time.time()
pycolmap.extract_features(DB_PATH, IMAGE_DIR, extraction_options=extraction_options)
print(f"feature extraction: {time.time()-t0:.1f}s")

t0 = time.time()
pycolmap.match_sequential(DB_PATH)
print(f"sequential matching: {time.time()-t0:.1f}s")

## 3. Incremental mapping, with the registration order captured live

Two callbacks, found by reading `colmap/src/colmap/controllers/incremental_pipeline.cc`
(search for `Callback(`): `INITIAL_IMAGE_PAIR_REG_CALLBACK` fires once for the seed
pair (2 images at once), `NEXT_IMAGE_REG_CALLBACK` fires once per image after that.
Both are wired to the same handler below, which diffs the reconstruction's currently-
registered image set against what it saw last time to identify exactly which image(s)
just got added -- `IncrementalPipelineCallback` values only signal "something changed,"
they don't pass the image identity directly.

In [ ]:
registration_log = []
step_counter = {"n": 0}
seen_ids = set()

db = pycolmap.Database.open(DB_PATH)
image_id_to_name = {img.image_id: img.name for img in db.read_all_images()}

db_cache = pycolmap.DatabaseCache.create(
    db, pycolmap.DatabaseCacheOptions(min_num_matches=15, ignore_watermarks=False)
)
recon_manager = pycolmap.ReconstructionManager()
mapper_options = pycolmap.IncrementalPipelineOptions()
mapper_options.image_path = str(IMAGE_DIR)
pipeline = pycolmap.IncrementalPipeline(mapper_options, db_cache, recon_manager)

t_start = time.time()

def on_registration_event():
    global seen_ids
    step_counter["n"] += 1
    if recon_manager.size() == 0:
        return
    recon = recon_manager.get(recon_manager.size() - 1)
    reg_ids = set(int(i) for i in recon.reg_image_ids())
    new_ids = sorted(reg_ids - seen_ids)
    seen_ids = reg_ids
    for nid in new_ids:
        registration_log.append({
            "step": step_counter["n"],
            "elapsed_s": round(time.time() - t_start, 3),
            "image_id": nid,
            "image_name": image_id_to_name.get(nid, "?"),
            "num_registered_so_far": len(reg_ids),
            "num_points3D": recon.num_points3D(),
        })
    if new_ids:
        print(f"  +{[image_id_to_name.get(n,'?') for n in new_ids]} "
              f"-> {len(reg_ids)} registered, {recon.num_points3D()} points3D "
              f"@ t={time.time()-t_start:.1f}s")

pipeline.add_callback(pycolmap.IncrementalPipelineCallback.INITIAL_IMAGE_PAIR_REG_CALLBACK, on_registration_event)
pipeline.add_callback(pycolmap.IncrementalPipelineCallback.NEXT_IMAGE_REG_CALLBACK, on_registration_event)
pipeline.run()

print(f"\nmapping done in {time.time()-t_start:.1f}s")
assert recon_manager.size() > 0, "reconstruction failed -- see the overlap note in §1"
reconstruction = recon_manager.get(0)
print(f"{len(reconstruction.reg_image_ids())} images registered, {reconstruction.num_points3D()} points3D")
print(f"registration_log: {len(registration_log)} entries")

## 4. Export point cloud + camera poses + registration order to the website

Same binary format (`Float32` positions, `Uint8` colors) and manifest structure as
`gaussian_point_cloud_visualization.ipynb`, so this dataset shows up in the exact same
WebGL viewer/dropdown -- with one addition, `registration_order`, a small JSON list
the viewer's replay slider reads to reveal cameras in the order COLMAP actually
registered them, not file order.

In [ ]:
import json
import numpy as np
import matplotlib as mpl
import matplotlib.colors as mcolors

def write_point_bin(name, xyz, rgb_u8):
    pos_path = DATA_DIR / f"{name}_pos.bin"
    col_path = DATA_DIR / f"{name}_col.bin"
    pos_path.write_bytes(xyz.astype(np.float32).tobytes())
    col_path.write_bytes(rgb_u8.astype(np.uint8).tobytes())
    return {"n": int(len(xyz)), "pos": pos_path.name, "col": col_path.name}

def write_camera_bin(name, pos, fwd, names):
    pos_path = DATA_DIR / f"{name}_cam_pos.bin"
    fwd_path = DATA_DIR / f"{name}_cam_fwd.bin"
    pos_path.write_bytes(pos.astype(np.float32).tobytes())
    fwd_path.write_bytes(fwd.astype(np.float32).tobytes())
    return {"n": int(len(pos)), "pos": pos_path.name, "fwd": fwd_path.name, "image_names": list(names)}

def robust_bbox(xyz, lo=2.0, hi=98.0):
    return np.percentile(xyz, lo, axis=0), np.percentile(xyz, hi, axis=0)

# --- point cloud from the final reconstruction ---
points = list(reconstruction.points3D.values())
xyz = np.array([p.xyz for p in points], dtype=np.float32)
rgb = np.array([p.color for p in points], dtype=np.uint8)
final_entry = write_point_bin(f"{DATASET_KEY}_final", xyz, rgb)
print(f"points3D exported: {len(points)} -> {final_entry['n']}")

# --- camera poses: position = projection_center(), forward = viewing_direction() ---
images = list(reconstruction.images.values())
cam_pos = np.array([img.projection_center() for img in images], dtype=np.float64)
cam_fwd = np.array([img.viewing_direction() for img in images], dtype=np.float64)
cam_names = [img.name for img in images]
camera_entry = write_camera_bin(DATASET_KEY, cam_pos, cam_fwd, cam_names)
print(f"cameras exported: {len(images)}")

bbox_lo, bbox_hi = robust_bbox(xyz)

# --- registration order, as its own small JSON (not worth a binary format) ---
regorder_path = DATA_DIR / f"{DATASET_KEY}_regorder.json"
with open(regorder_path, "w") as f:
    json.dump(registration_log, f)
registration_order_entry = {"n": len(registration_log), "path": regorder_path.name}
print(f"registration order exported: {len(registration_log)} steps -> {regorder_path.name}")

In [ ]:
manifest_path = DATA_DIR / "manifest.json"
manifest = json.load(open(manifest_path)) if manifest_path.exists() else {"datasets": {}}

manifest["datasets"][DATASET_KEY] = {
    "label": DATASET_LABEL,
    # no "initial" entry -- COLMAP's OWN sparse output IS the "final" here; there is no
    # separate densified/trained stage the way there is for the pipeline-results Gaussian
    # runs, so the growth-voxel layer doesn't apply to this dataset either.
    "final": final_entry,
    "cameras": camera_entry,
    "registration_order": registration_order_entry,
    "bbox_min": bbox_lo.tolist(),
    "bbox_max": bbox_hi.tolist(),
    "legend_html": (
        f"<b>{DATASET_LABEL}</b><br>"
        f"COLMAP sparse reconstruction from {NUM_IMAGES} real photos "
        f"({SCENE_IMAGES_DIR.parent.name})<br>"
        f"points3D: {len(points):,}<br>"
        f"cameras registered: {len(images)}<br>"
        f"use the replay slider in the panel to step through registration order"
    ),
}
with open(manifest_path, "w") as f:
    json.dump(manifest, f)
print(f"manifest.json updated -- {len(manifest['datasets'])} datasets total (including this one)")

## 5. Make sure the viewer HTML is up to date

If you've never run `gaussian_point_cloud_visualization.ipynb` in this checkout,
`viz/index.html` (with the registration-replay control) may not exist yet -- this
writes it if missing, without touching it if it's already there (so re-running this
cell after editing the viewer by hand doesn't clobber your edits).

In [ ]:
index_html_path = VIZ_ROOT / "index.html"
if index_html_path.exists():
    print("viz/index.html already exists, leaving it as-is:", index_html_path.stat().st_size, "bytes")
else:
    raise SystemExit(
        "viz/index.html is missing. Run gaussian_point_cloud_visualization.ipynb once "
        "first (it writes the full WebGL viewer, including the registration-replay "
        "control this notebook's export relies on), then re-run this notebook."
    )

## 6. Start the local server

In [ ]:
import subprocess, sys, urllib.request

def start_server(directory, ports=range(8000, 8010)):
    for port in ports:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}", timeout=0.2)
            continue
        except Exception:
            pass
        proc = subprocess.Popen(
            [sys.executable, "-m", "http.server", str(port), "--directory", str(directory)],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        time.sleep(0.6)
        if proc.poll() is None:
            return proc, port
    raise RuntimeError(f"could not bind any port in {ports}")

_server_proc, _server_port = start_server(VIZ_ROOT)
print(f"Serving {VIZ_ROOT} at http://localhost:{_server_port}/")
print(f"Pick '{DATASET_LABEL}' from the dropdown, then use the 'COLMAP registration replay' "
      f"slider/Play button in the panel to watch the images get registered in the exact "
      f"order COLMAP chose them.")

## 7. Stop the server (run when done viewing)

In [ ]:
_server_proc.terminate()
_server_proc.wait(timeout=5)
print("server stopped")

## Summary

- This notebook runs the real COLMAP pipeline (`pycolmap`: `extract_features` ->
  `match_sequential` -> `IncrementalPipeline`) on training images from `dataset/`,
  rather than only visualizing a `.ply` a previous run already produced.
- The camera **registration order** -- which image COLMAP chose to register at each
  step, not file order -- is captured live via two callbacks found in
  `colmap/src/colmap/controllers/incremental_pipeline.cc`
  (`INITIAL_IMAGE_PAIR_REG_CALLBACK`, `NEXT_IMAGE_REG_CALLBACK`) and exported as
  `<key>_regorder.json`.
- The website viewer (`viz/index.html`) has a new "COLMAP registration replay"
  slider/Play control that uses this log to reveal camera frustums in the exact order
  they were registered -- built specifically to "re-trace/watch again" the process, as
  asked.
- Validated end-to-end on 40 real photos from `mipnerf360/garden`: full reconstruction
  (40/40 registered, 29k+ points3D) in ~7.5 minutes on CPU only (no CUDA available on
  this machine). An evenly-spread subsample across the whole capture was tried first
  and failed to grow past a few images -- use a **consecutive** subsequence (§1),
  which preserves the frame-to-frame overlap sequential matching depends on.
- To run a different scene or more images, edit `SCENE_IMAGES_DIR`/`NUM_IMAGES` in the
  Config cell and re-run from the top; budget CPU time roughly per the §Intro timing
  data point, or run on a GPU machine for full-size scenes.